<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase3-llm-evaluation/03_llm_as_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: LLM as a Judge
Goal: Use an LLM to automatically score and evaluate LLM responses
Date: May 2026.
Status: In Progress

In [4]:
import time
import pandas as pd
import json
import re
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Function 1 - Generate response with retry
def ask_llm(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model="gemini-flash-latest",
                contents=prompt
            )
            return response.text
        except Exception as e:
            print(f"     Attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return "Error: max retries exceeded"

# Function 2 - Judge with retry
def judge_response(question, response, retries=3):
    judge_prompt = f"""
You are an expert AI governance evaluator.
Score the following response on three criteria.
Return ONLY a JSON object, nothing else.
No markdown, no backticks, just the raw JSON.

Question asked: {question}

Response to evaluate: {response}

Score each criterion from 1-5:
- accuracy: Is the information factually correct?
- clarity: Is the response clear and well structured?
- relevance: Does the response directly address the question?

Also provide a one sentence summary of your evaluation.

Return exactly this format:
{{
  "accuracy": <score>,
  "clarity": <score>,
  "relevance": <score>,
  "summary": "<one sentence>"
}}
"""
    for attempt in range(retries):
        try:
            result = client.models.generate_content(
                model="gemini-flash-latest",
                contents=judge_prompt
            )
            return result.text
        except Exception as e:
            print(f"     Judge attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return "Error: max retries exceeded"

# Function 3 - Parse JSON robustly
def parse_judgment(raw):
    # Remove markdown, extra whitespace
    clean = re.sub(r"```json|```", "", raw).strip()
    # Find JSON object even if surrounded by text
    match = re.search(r'\{.*\}', clean, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"No JSON found in: {raw}")

# Test prompts
test_prompts = [
    "What is AI governance in two sentences?",
    "What are the main risks of unregulated AI?",
    "Why does the EU AI Act matter for businesses?",
]

# Run eval and judge
results = []
print("====== LLM AS JUDGE EVALUATION ======\n")

for i, prompt in enumerate(test_prompts):
    print(f"[{i+1}] Generating response...")
    response = ask_llm(prompt)
    time.sleep(2)

    print(f"[{i+1}] Judging response...")
    judgment = judge_response(prompt, response)
    time.sleep(2)

    try:
        scores = parse_judgment(judgment)
        results.append({
            "prompt_number": i + 1,
            "prompt":        prompt,
            "response":      response,
            "accuracy":      scores["accuracy"],
            "clarity":       scores["clarity"],
            "relevance":     scores["relevance"],
            "summary":       scores["summary"],
        })
        print(f"     Accuracy:  {scores['accuracy']}/5")
        print(f"     Clarity:   {scores['clarity']}/5")
        print(f"     Relevance: {scores['relevance']}/5")
        print(f"     Summary:   {scores['summary']}\n")

    except Exception as e:
        print(f"     Parse error: {e}")
        print(f"     Raw: {judgment}\n")

# Build dataframe
df = pd.DataFrame(results)

print("====== JUDGMENT SUMMARY ======")
print(f"Total evaluated:  {len(df)}")
print(f"Avg Accuracy:     {df['accuracy'].mean():.1f}/5")
print(f"Avg Clarity:      {df['clarity'].mean():.1f}/5")
print(f"Avg Relevance:    {df['relevance'].mean():.1f}/5")

# Save
df.to_csv(SAVE_PATH + "llm_as_judge_results.csv", index=False)
print("\nResults saved ✅")

df[["prompt_number", "prompt", "accuracy", "clarity", "relevance", "summary"]]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
====== LLM AS JUDGE EVALUATION ======

[1] Generating response...
[1] Judging response...
     Accuracy:  5/5
     Clarity:   5/5
     Relevance: 5/5
     Summary:   The response provides a concise, accurate, and comprehensive definition of AI governance while perfectly adhering to the two-sentence constraint.

[2] Generating response...
[2] Judging response...
     Accuracy:  5/5
     Clarity:   5/5
     Relevance: 5/5
     Summary:   The response provides a comprehensive, well-structured, and accurate overview of the multifaceted risks associated with unregulated AI, covering social, economic, and technical dimensions.

[3] Generating response...
[3] Judging response...
     Accuracy:  5/5
     Clarity:   5/5
     Relevance: 5/5
     Summary:   The response is a factually precise, well-structured, and highly relevant summary of the EU AI Act's impact on bus

,prompt_number,prompt,accuracy,clarity,relevance,summary
0,1,What is AI governance in two sentences?,5,5,5,"The response provides a concise, accurate, and..."
1,2,What are the main risks of unregulated AI?,5,5,5,"The response provides a comprehensive, well-st..."
2,3,Why does the EU AI Act matter for businesses?,5,5,5,"The response is a factually precise, well-stru..."


## Findings: LLM as Judge Evaluation

**Model tested:** gemini-flash-latest
**Judge model:** gemini-flash-latest
**Date:** May 2026
**Prompts evaluated:** 3

### Results

| Prompt | Accuracy | Clarity | Relevance |
|--------|----------|---------|-----------|
| AI Governance definition | 5/5 | 5/5 | 5/5 |
| Risks of unregulated AI | 5/5 | 5/5 | 5/5 |
| EU AI Act for businesses | 5/5 | 5/5 | 5/5 |

### Key Finding: Judge Bias Detected

All responses scored perfect 5/5 across all dimensions.
This is a signal of leniency bias, the judge model
is evaluating responses from the same model family
too favorably.

This is a known problem in LLM evaluation called
"self-serving bias". Models tend to rate outputs
from their own family higher than outputs from
competing models.

### Recommendation
In production eval systems always use a different
model family as the judge. Example:
- Response generated by: Gemini
- Response judged by: Claude or GPT-4

This cross-model judging produces more objective scores.

### What Good Scores Look Like
A well calibrated judge should show variance:
- Strong responses: 4-5/5
- Average responses: 3/5
- Weak responses: 1-2/5

A judge that always scores 5/5 is not useful
as a quality signal.